In [13]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# LSTM Lyrics Classifier

This notebook builds a full text classification workflow for lyrics using an LSTM model:

1. Load the lyrics dataset.
2. Clean and normalize the text.
3. Tokenize and build a vocabulary from the training split.
4. Convert lyrics into padded integer sequences.
5. Train and evaluate an LSTM classifier in PyTorch.

In [17]:
data_path = Path(r"C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")
df = pd.read_csv(data_path)
df = df[["verse", "label"]].dropna().drop_duplicates().reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts().sort_index())

(22878, 2)
label
0    11439
1    11439
Name: count, dtype: int64


#### Text Cleaning and Train/Validation/Test Split

The LSTM should only see text from the training split when building the vocabulary. That avoids leaking information from validation or test lyrics into the tokenizer.


In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text) # Replace non-alphanumeric characters with spaces
    text = re.sub(r"\s+", " ", text).strip() # Replace multiple spaces with a single space and trim
    return text

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=seed,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=seed,
    stratify=temp_df["label"],
)

for frame in (train_df, val_df, test_df):
    frame.loc[:, "clean_verse"] = frame["verse"].apply(clean_text)

print("train:", train_df.shape, train_df["label"].value_counts().sort_index().to_dict())
print("val:", val_df.shape, val_df["label"].value_counts().sort_index().to_dict())
print("test:", test_df.shape, test_df["label"].value_counts().sort_index().to_dict())

train: (18302, 3) {0: 9151, 1: 9151}
val: (2288, 3) {0: 1144, 1: 1144}
test: (2288, 3) {0: 1144, 1: 1144}


In [7]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
MIN_FREQ = 2
MAX_LEN = 180


def tokenize(text: str) -> list[str]:
    return text.split()


counter = Counter()
for text in train_df["clean_verse"]:
    counter.update(tokenize(text))

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[token] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")
print("Most common tokens:", counter.most_common(10))

Vocabulary size: 12424
Most common tokens: [('i', 54847), ('you', 40504), ('the', 31054), ('to', 24190), ('it', 20204), ('a', 18599), ('me', 18295), ('and', 17795), ('not', 15736), ('is', 15135)]


In [8]:
def numericalize(text: str) -> tuple[torch.Tensor, torch.Tensor]:
    token_ids = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text)]
    length = min(len(token_ids), MAX_LEN)
    token_ids = token_ids[:MAX_LEN]
    if len(token_ids) < MAX_LEN:
        token_ids += [vocab[PAD_TOKEN]] * (MAX_LEN - len(token_ids))
    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(length, dtype=torch.long)

sample_ids, sample_length = numericalize(train_df.iloc[0]["clean_verse"])
print("sample length:", sample_length.item())
print("sample ids:", sample_ids[:20].tolist())

sample length: 46
sample ids: [2, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 10, 11, 12, 13, 14, 14, 15]


In [9]:
class LyricsDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.texts = frame["clean_verse"].tolist()
        self.labels = frame["label"].astype(np.float32).tolist()

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int):
        input_ids, length = numericalize(self.texts[index])
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        return input_ids, length, label


batch_size = 64
train_loader = DataLoader(LyricsDataset(train_df), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(LyricsDataset(val_df), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(LyricsDataset(test_df), batch_size=batch_size, shuffle=False)

batch_input_ids, batch_lengths, batch_labels = next(iter(train_loader))
print(batch_input_ids.shape, batch_lengths.shape, batch_labels.shape)

torch.Size([64, 180]) torch.Size([64]) torch.Size([64])


LSTM Model Definition

The model uses an embedding layer, an LSTM encoder, dropout, and a linear output layer for binary classification.


In [10]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 128,
        hidden_dim: int = 128,
        num_layers: int = 2,
        bidirectional: bool = True,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab[PAD_TOKEN])
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(output_dim, 1)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed)
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        logits = self.fc(self.dropout(hidden))
        return logits.squeeze(1)


model = LSTMClassifier(vocab_size=len(vocab)).to(device)
print(model)

LSTMClassifier(
  (embedding): Embedding(12424, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [11]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def run_epoch(loader: DataLoader, training: bool = True):
    model.train() if training else model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(training):
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predictions = (torch.sigmoid(logits) >= 0.5).long()
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().long().tolist())
        total_loss += loss.item() * input_ids.size(0)

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, accuracy, all_targets, all_predictions


with torch.no_grad():
    preview_logits = model(batch_input_ids.to(device), batch_lengths.to(device))
print("preview logits shape:", preview_logits.shape)

preview logits shape: torch.Size([64])


In [14]:
num_epochs = 5
best_val_loss = float("inf")
best_model_path = Path("lstm_lyrics_classifier.pt")
history = []

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, training=False)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
history_df

Epoch 01 | train_loss=0.5474 train_acc=0.7308 | val_loss=0.5624 val_acc=0.7404
Epoch 02 | train_loss=0.5109 train_acc=0.7493 | val_loss=0.5050 val_acc=0.7631
Epoch 03 | train_loss=0.4622 train_acc=0.7832 | val_loss=0.4785 val_acc=0.7727
Epoch 04 | train_loss=0.4003 train_acc=0.8204 | val_loss=0.4484 val_acc=0.7968
Epoch 05 | train_loss=0.3431 train_acc=0.8523 | val_loss=0.4184 val_acc=0.8164


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.547384,0.730794,0.562401,0.740385
1,2,0.510899,0.749317,0.505043,0.763112
2,3,0.462157,0.783193,0.478547,0.772727
3,4,0.400273,0.820402,0.448413,0.796766
4,5,0.343092,0.852311,0.418441,0.816434


Final Evaluation and Saving

After training, reload the best checkpoint and evaluate it once on the test split. Then save the model and vocabulary together for inference.

In [15]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc, test_targets, test_predictions = run_epoch(test_loader, training=False)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(test_targets, test_predictions, digits=4))

Test loss: 0.4337
Test accuracy: 0.7885
              precision    recall  f1-score   support

           0     0.8005    0.7684    0.7841      1144
           1     0.7773    0.8086    0.7926      1144

    accuracy                         0.7885      2288
   macro avg     0.7889    0.7885    0.7884      2288
weighted avg     0.7889    0.7885    0.7884      2288



In [16]:
artifact_dir = Path("lstm_artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "lyrics_lstm.pt"
vocab_path = artifact_dir / "lyrics_vocab.json"

torch.save(model.state_dict(), model_path)
with open(vocab_path, "w", encoding="utf-8") as vocab_file:
    json.dump(vocab, vocab_file, ensure_ascii=False, indent=2)


def predict_text(text: str):
    model.eval()
    cleaned_text = clean_text(text)
    input_ids, length = numericalize(cleaned_text)
    with torch.no_grad():
        logits = model(input_ids.unsqueeze(0).to(device), length.unsqueeze(0).to(device))
        probability = torch.sigmoid(logits).item()
        prediction = int(probability >= 0.5)
    return probability, prediction


sample_probability, sample_prediction = predict_text("i will always love you")
print({"probability": sample_probability, "prediction": sample_prediction})

{'probability': 0.036332253366708755, 'prediction': 0}
